# Lab 5: FIR Filter -- MMIO vs DMA Streaming

This notebook compares two approaches for running an FIR filter on the Pynq FPGA:
1. **MMIO**: CPU sends one sample at a time via AXI-Lite (slow)
2. **DMA Streaming**: CPU sets up a bulk transfer, HW processes autonomously (fast)

Both use the same 21-tap low-pass filter.

## 1. Load Overlay

In [2]:
import time
import struct
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("matmulfloat.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
matmul_stream  = ol.matmul_stream_1_0
fir_mmio_ip = ol.fir_mmio_0
hw_timer    = ol.axi_timer_0

Overlay loaded
IP blocks: ['matmul_stream_1_0', 'fir_mmio_0', 'axi_timer_0', 'axi_dma_0', 'ps7']


## 2. Prepare Test Signal

50 Hz (passband) + 300 Hz (stopband), sampled at 1 kHz.

In [3]:
import sys
import os
import numpy as np

# -----------------------------
# Normalization helpers
# -----------------------------
def normalize_vector(x):
    norm = np.linalg.norm(x)
    return x if norm == 0 else x / norm

def normalize_matrix(A):
    norm = np.linalg.norm(A)
    return A if norm == 0 else A / norm

# -----------------------------
# Generate vector
# -----------------------------
def generate_vector(n, save = True):
    os.makedirs("test/input", exist_ok=True)

    X = np.random.randn(n, 1)
    X = normalize_vector(X)

    if save:
        path = f"test/input/{n}.txt"
        np.savetxt(path, X, fmt="%.6f")

        print(f"Vector saved to {path}")
    return X

# -----------------------------
# Generate matrix
# -----------------------------
def generate_matrix(n, save = True):
    os.makedirs("test/matrices", exist_ok=True)

    A = np.random.randn(n, n)
    A = normalize_matrix(A)

    if save:
        path = f"test/matrices/{n}.txt"
        np.savetxt(path, A, fmt="%.6f")

        print(f"Matrix saved to {path}")
    return A

# -----------------------------
# Main pipeline
# -----------------------------
def generate_all(n, save = True):
    # Generate inputs
    A = generate_matrix(n, save)
    X = generate_vector(n, save)

    # Compute Y = A * X
    Y = np.matmul(A, X)

    if save:
        os.makedirs("test/output", exist_ok=True)
        out_path = f"test/output/{n}.txt"
        np.savetxt(out_path, Y, fmt="%.6f")

        print(f"Output (Y = AX) saved to {out_path}")
    return A, X, Y



In [4]:
import numpy as np

N = 10
matrix_data, input_vector, output_vector = generate_all(N, False)

# Convert list → numpy array
matrix_data = np.array(matrix_data, dtype=np.float32)
input_vector = np.array(input_vector, dtype=np.float32)

print(matrix_data.shape)
print(input_vector.shape)

N_SAMPLES = N*N + N   # matrix + vector

# Reshape into proper matrix
matrix = matrix_data.reshape(N, N)

# Ensure vector shape
x = input_vector.reshape(N,)   # or (N,1) if needed

# Flatten matrix and concatenate
matrix_flat = matrix.flatten()
full_input = np.concatenate([matrix_flat, x]).astype(np.float32)
# FS = 1000.0
# t = np.arange(N_SAMPLES) / FS
# x = (0.7 * np.sin(2 * np.pi * 50.0 * t) +
#      0.3 * np.sin(2 * np.pi * 300.0 * t)).astype(np.float32)
# x[0] += 1.0

# print(f"Input: {N_SAMPLES} samples, dtype={x.dtype}")

(10, 10)
(10, 1)


## 3. MMIO FIR (one sample per round-trip)

Each sample requires: write input $\rightarrow$ start IP $\rightarrow$ poll done $\rightarrow$ read output.

**Note:** Check the register offsets against the HLS driver header
(`xfir_mmio_hw.h`) after synthesis.

In [ ]:
# # Register offsets (from HLS synthesis report -- verify after build)
# CTRL_REG       = 0x00
# X_IN_OFFSET    = 0x18
# RETURN_OFFSET  = 0x10

# def float_to_uint(f):
#     return struct.unpack('<I', struct.pack('<f', f))[0]

# def uint_to_float(u):
#     return struct.unpack('<f', struct.pack('<I', u & 0xFFFFFFFF))[0]

# def mmio_fir_one_sample(ip, x_val):
#     ip.write(X_IN_OFFSET, float_to_uint(x_val))
#     ip.write(CTRL_REG, 0x01)                    # ap_start
#     while (ip.read(CTRL_REG) & 0x02) == 0:      # wait ap_done
#         pass
#     return uint_to_float(ip.read(RETURN_OFFSET))

In [ ]:
# y_mmio = np.zeros(N_SAMPLES, dtype=np.float32)

# t0 = time.perf_counter()y_mmio = np.zeros(N_SAMPLES, dtype=np.float32)

# t0 = time.perf_counter()
# for i in range(N_SAMPLES):
#     y_mmio[i] = mmio_fir_one_sample(fir_mmio_ip, float(x[i]))
# t_mmio = time.perf_counter() - t0

# print(f"MMIO: {N_SAMPLES} samples in {t_mmio*1e3:.2f} ms "
#       f"({t_mmio/N_SAMPLES*1e6:.1f} us/sample)")
# for i in range(N_SAMPLES):
#     y_mmio[i] = mmio_fir_one_sample(fir_mmio_ip, float(x[i]))
# t_mmio = time.perf_counter() - t0

# print(f"MMIO: {N_SAMPLES} samples in {t_mmio*1e3:.2f} ms "
#       f"({t_mmio/N_SAMPLES*1e6:.1f} us/sample)")

## 4. DMA Streaming FIR

Data path: PS $\rightarrow$ DMA TX $\rightarrow$ AXI-Stream $\rightarrow$ FIR $\rightarrow$ AXI-Stream $\rightarrow$ DMA RX $\rightarrow$ PS

The CPU only sets up the transfer and waits -- the FPGA processes all samples autonomously.

In [14]:
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.float32)
out_buf = allocate(shape=(10,), dtype=np.float32)

np.copyto(in_buf, full_input)
out_buf[:] = 0

# Configure streaming FIR: set sample count and start
# Register 0x10 = 'n' parameter (check synthesis report)
matmul_stream.write(0x10, 0x01)
matmul_stream.write(0x00, 0x01)   # ap_start

t0 = time.perf_counter()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

y_dma = np.array(out_buf, dtype=np.float32)
print(f"DMA: {N_SAMPLES} samples in {t_dma*1e3:.2f} ms "
      f"({t_dma/N_SAMPLES*1e6:.1f} us/sample)")

DMA: 110 samples in 3.46 ms (31.4 us/sample)


In [18]:
# y_dma = y_dma[1:10]
print(y_dma)
y_calculated = np.reshape(y_dma, (N, 1))
y_calculated-output_vector

[ 0.04889314  0.02057634  0.19197585  0.14377154  0.0441099  -0.04303066
 -0.1441982   0.02464324  0.01148681  0.02662289]


array([[-1.54520566e-09],
       [ 1.10459137e-09],
       [-7.48606360e-09],
       [-5.22987295e-09],
       [ 9.39678677e-09],
       [ 1.01299300e-08],
       [-1.36528621e-08],
       [-8.16437758e-10],
       [-2.49108869e-09],
       [-3.21071354e-09]])

## 5. AXI Timer (cycle-accurate HW timing)

The AXI Timer counts PL clock cycles. This measures only the
HW processing time (including DMA transfer), without Python overhead.

In [ ]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles

In [ ]:
# Re-run DMA test with HW timer
np.copyto(in_buf, x)
out_buf[:] = 0

fir_stream.write(0x10, N_SAMPLES)
fir_stream.write(0x00, 0x01)

timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

print(f"HW timer: {cycles} cycles = {cycles/FCLK_MHZ:.1f} us @ {FCLK_MHZ:.0f} MHz")

## 6. Comparison

In [ ]:
print(f"{'Method':<12} {'Time (ms)':>10} {'Speedup':>10}")
print("-" * 34)
print(f"{'MMIO':<12} {t_mmio*1e3:>10.2f} {'1.0x':>10}")
print(f"{'DMA':<12} {t_dma*1e3:>10.2f} {t_mmio/t_dma:>9.1f}x")
print(f"{'HW cycles':<12} {cycles/FCLK_MHZ/1e3:>10.3f} {'(PL only)':>10}")
print()

max_diff = np.max(np.abs(y_dma - y_mmio))
print(f"Max |DMA - MMIO| = {max_diff:.2e}")
assert max_diff < 1e-4, "Output mismatch!"

## 7. Plot Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t * 1e3, x, label="Input (50+300 Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("FIR Filter Input")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1e3, y_dma, label="DMA output")
axes[1].plot(t * 1e3, y_mmio, '--', label="MMIO output", alpha=0.7)
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("FIR Filter Output (300 Hz suppressed)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Cleanup

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()